# Redox FHIR Pipeline - Discover Resource Types

This notebook discovers all available FHIR resource types and sets task values for parallel Silver table creation in a Databricks workflow.

## Purpose
This notebook is designed to run as part of a Databricks workflow to:
1. Query the `resource_schemas` table for distinct resource types
2. Set task values that drive a for-each loop for Silver table creation

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT 'bronze';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE schema_use = COALESCE(:schema_use, schema_use);

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog() AS catalog, current_schema() AS schema;

In [ ]:
-- Get all distinct resource types
SELECT DISTINCT resource_type
FROM resource_schemas
ORDER BY resource_type;

In [ ]:
%python
# Set task values for workflow for-each loop
resource_types_df = spark.sql("""
    SELECT DISTINCT resource_type 
    FROM resource_schemas 
    ORDER BY resource_type
""")

resource_types = [row.resource_type for row in resource_types_df.collect()]

print(f"Discovered {len(resource_types)} resource types:")
for rt in resource_types:
    print(f"  - {rt}")

# Set task value for downstream for-each task
dbutils.jobs.taskValues.set("resource_types", resource_types)

In [ ]:
-- Resource type summary with field counts
SELECT 
  resource_type,
  COUNT(*) AS field_count
FROM resource_schemas
GROUP BY resource_type
ORDER BY field_count DESC;